# RL for Optimal Trade Execution

A PPO agent that makes per-second execution decisions based on the current order book state.
Instead of predicting 60 seconds ahead, the agent reacts to what it sees right now.

In [1]:
import os, sys, subprocess

# Colab setup
if os.path.exists("/content"):
    # Always ensure we're in the right directory
    repo_dir = "/content/Ultramarin"

    # Clone repo if needed (for simulate_walk_the_book.py)
    if not os.path.isdir(os.path.join(repo_dir, "utils")):
        subprocess.run(["git", "clone", "-q", "-b", "potentially-improve-model",
                        "https://github.com/JosephZacharyGawlik/Ultramarin.git"],
                       cwd="/content")

    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

    # Mount Drive and copy data if needed
    dst = os.path.join(repo_dir, "data")
    if not os.path.isdir(dst) or not any(
        d for d in os.listdir(dst) if os.path.isdir(os.path.join(dst, d))
    ):
        from google.colab import drive
        drive.mount('/content/drive')
        src = "/content/drive/MyDrive/data"
        os.makedirs(dst, exist_ok=True)
        subprocess.run(["cp", "-r", f"{src}/.", dst], capture_output=True, text=True)
        print("Data copied.")
    else:
        print(f"Data already present at {dst}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "gymnasium", "stable-baselines3"], capture_output=True)
print("Setup complete.")

Data already present at /content/Ultramarin/data
Setup complete.


In [2]:
import numpy as np
import pandas as pd
import math
import warnings
import random
from pathlib import Path

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

from data.simulate_walk_the_book import simulate_walk_the_book

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# === Change this to switch currencies ===
data_asset = "DOGEUSDT"

KNOWN_VOLUMES = {
    "BTCUSDT": 4.0, "ETHUSDT": 55.0, "LTCUSDT": 51.0,
    "SOLUSDT": 315.0, "ADAUSDT": 10436.0, "DOGEUSDT": 60349.0, "XRPUSDT": 13736.0,
}
OPTIMAL_K = {
    "BTCUSDT": 14, "ETHUSDT": 26, "LTCUSDT": 16,
    "SOLUSDT": 17, "ADAUSDT": 7, "DOGEUSDT": 20, "XRPUSDT": 20,
}

volume_to_fill = KNOWN_VOLUMES[data_asset]
K_SECONDS = OPTIMAL_K[data_asset]

root = Path("data")
Y_path = root / data_asset / "y_train.parquet"

print(f"Currency: {data_asset}")
print(f"Volume to fill: {volume_to_fill}")
print(f"TWAP-K baseline: K={K_SECONDS}")

## Data Loading

Load Y data (the 60-second execution window), split into train/val using the same
chrono split as mhf-final (sort IDs, 80/20 split), and pre-extract LOB arrays per instrument.

In [4]:
ASK_PRICE_COLS = [f'ask_price_{i}' for i in range(1, 6)]
ASK_VOL_COLS = [f'ask_vol_{i}' for i in range(1, 6)]
BID_PRICE_COLS = [f'bid_price_{i}' for i in range(1, 6)]
BID_VOL_COLS = [f'bid_vol_{i}' for i in range(1, 6)]

Y_raw = pd.read_parquet(Y_path).sort_values(["anonymized_id", "time_in_hour"])

# Clean prices and volumes the same way as mhf-final:
# 1. Forward-fill then backfill prices (handles NaN at start AND middle)
# 2. Zero-fill volumes
price_cols = ASK_PRICE_COLS + BID_PRICE_COLS
vol_cols = ASK_VOL_COLS + BID_VOL_COLS
Y_raw[price_cols] = Y_raw.groupby("anonymized_id")[price_cols].transform(
    lambda s: s.ffill().bfill()
)
Y_raw[vol_cols] = Y_raw[vol_cols].fillna(0.0)

# Get valid instrument IDs (exactly 60 rows)
id_counts = Y_raw.groupby("anonymized_id").size()
valid_ids = sorted(id_counts[id_counts == 60].index.tolist())

# Chrono split: same logic as train_val() — sort IDs, take last 20% as val
split_idx = int(len(valid_ids) * 0.8)
train_ids = valid_ids[:split_idx]
val_ids = valid_ids[split_idx:]

print(f"Valid instruments: {len(valid_ids)}")
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}")


def extract_instruments(ids, df):
    """Pre-extract LOB arrays for each instrument."""
    instruments = []
    skipped = 0
    for aid in ids:
        df_inst = df[df["anonymized_id"] == aid].sort_values("time_in_hour")
        if len(df_inst) != 60:
            continue

        ask_prices = df_inst[ASK_PRICE_COLS].to_numpy(dtype=np.float64)
        bid_prices = df_inst[BID_PRICE_COLS].to_numpy(dtype=np.float64)
        ask_vols = df_inst[ASK_VOL_COLS].to_numpy(dtype=np.float64)
        bid_vols = df_inst[BID_VOL_COLS].to_numpy(dtype=np.float64)

        # Skip if level-1 prices are still NaN (entire column was NaN — no data at all)
        if np.isnan(ask_prices[:, 0]).any() or np.isnan(bid_prices[:, 0]).any():
            skipped += 1
            continue

        close_vals = df_inst["close"].dropna()
        if len(close_vals) == 0:
            skipped += 1
            continue

        instruments.append({
            "id": aid,
            "ask_prices": ask_prices,
            "ask_vols": ask_vols,
            "bid_prices": bid_prices,
            "bid_vols": bid_vols,
            "close": close_vals.iloc[-1],
        })
    if skipped > 0:
        print(f"  Skipped {skipped} instruments (all-NaN prices or missing close)")
    return instruments


train_instruments = extract_instruments(train_ids, Y_raw)
val_instruments = extract_instruments(val_ids, Y_raw)

print(f"Train instruments loaded: {len(train_instruments)}")
print(f"Val instruments loaded: {len(val_instruments)}")

Valid instruments: 652
Train: 521, Val: 131


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Train instruments loaded: 521
Val instruments loaded: 131


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## Gymnasium Environment

Each episode = one instrument's last K-second execution window (same window as TWAP-K and Oracle).  
State: 12 features (remaining vol/time, spread, depth, imbalance, momentum, etc.)  
Action: 6 discrete choices — multiples of the TWAP-K rate  

**Key design: agent only trades in the last K seconds.**  
This matches the Oracle's optimization window and TWAP-K's execution window.  
Executing earlier hurts BPS because prices drift from the close price.

**Reward (differential mark-to-market vs TWAP-K):**  
At each step, compare agent's cost-to-complete vs TWAP-K's cost-to-complete.  
Reward = change in advantage. Positive reward = agent is doing better than TWAP-K.

In [5]:
class TradeExecutionEnv(gym.Env):
    """
    RL environment for optimal trade execution in the last K seconds.

    Instead of trading across all 60 seconds (where price drift hurts BPS),
    the agent only trades in the last K seconds — the same window as TWAP-K
    and the Oracle optimizer. This makes the comparison fair and reduces noise.

    Reward: Differential mark-to-market advantage over TWAP-K.
    Action space: 10 discrete actions from 0x to 3x TWAP-K rate.
    """

    REWARD_SCALE = 10000  # puts rewards in ~bps units

    def __init__(self, instruments, volume_to_fill, K_seconds):
        super().__init__()
        self.instruments = instruments
        self.volume_to_fill = volume_to_fill
        self.K = K_seconds
        self.start_second = 60 - K_seconds
        self.twap_rate = volume_to_fill / K_seconds

        self.action_space = spaces.Discrete(10)
        self.action_multipliers = np.array([
            0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0
        ])

        self.observation_space = spaces.Box(
            low=-10.0, high=10.0, shape=(12,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        if options and "instrument_idx" in options:
            idx = options["instrument_idx"]
        else:
            idx = self.np_random.integers(len(self.instruments))
        self.inst = self.instruments[idx]

        self.t = 0
        self.volume_executed = 0.0
        self.value_executed = 0.0
        self.volume_allocated = 0.0
        self.remaining_carry = 0.0
        self.prev_advantage = 0.0

        self._precompute_twap()

        abs_t = self.start_second
        self.arrival_mid = (
            self.inst["ask_prices"][abs_t, 0] + self.inst["bid_prices"][abs_t, 0]
        ) / 2

        return self._get_obs(), {}

    def _precompute_twap(self):
        """Run TWAP-K through walk-the-book for the last K seconds."""
        self.twap_cost_per_step = np.zeros(self.K)
        self.twap_vol_per_step = np.zeros(self.K)
        carry = 0.0
        for step in range(self.K):
            abs_t = self.start_second + step
            to_fill = self.twap_rate + carry
            carry = 0.0
            filled = 0.0
            cost = 0.0
            remaining = to_fill
            for level in range(5):
                price = self.inst["ask_prices"][abs_t, level]
                avail = self.inst["ask_vols"][abs_t, level]
                if math.isnan(price) or math.isnan(avail):
                    continue
                take = min(remaining, avail)
                cost += take * price
                filled += take
                remaining -= take
                if remaining <= 1e-12:
                    break
            carry = remaining
            self.twap_cost_per_step[step] = cost
            self.twap_vol_per_step[step] = filled
        self.twap_cum_cost = np.cumsum(self.twap_cost_per_step)
        self.twap_cum_vol = np.cumsum(self.twap_vol_per_step)

    def _get_obs(self):
        if self.t >= self.K:
            return np.zeros(12, dtype=np.float32)

        abs_t = self.start_second + self.t
        ask_p = self.inst["ask_prices"][abs_t]
        ask_v = self.inst["ask_vols"][abs_t]
        bid_p = self.inst["bid_prices"][abs_t]
        bid_v = self.inst["bid_vols"][abs_t]
        mid = (ask_p[0] + bid_p[0]) / 2

        remaining_vol = max(0, self.volume_to_fill - self.volume_allocated)
        remaining_vol_frac = remaining_vol / (self.volume_to_fill + 1e-8)
        remaining_time_frac = (self.K - self.t) / self.K
        spread = (ask_p[0] - bid_p[0]) / (mid + 1e-8)
        mid_return = (mid - self.arrival_mid) / (self.arrival_mid + 1e-8)

        vol_imb_1 = (bid_v[0] - ask_v[0]) / (bid_v[0] + ask_v[0] + 1e-8)
        vol_imb_total = (bid_v.sum() - ask_v.sum()) / (bid_v.sum() + ask_v.sum() + 1e-8)
        ask_depth = ask_v.sum() / (self.volume_to_fill + 1e-8)
        bid_depth = bid_v.sum() / (self.volume_to_fill + 1e-8)

        cum_is = 0.0
        if self.volume_executed > 0:
            avg_price = self.value_executed / self.volume_executed
            cum_is = (avg_price - self.arrival_mid) / (self.arrival_mid + 1e-8)

        if abs_t >= 5:
            mid_prev = (self.inst["ask_prices"][abs_t-5, 0] + self.inst["bid_prices"][abs_t-5, 0]) / 2
            momentum = (mid - mid_prev) / (self.arrival_mid + 1e-8)
            prev_spread = (self.inst["ask_prices"][abs_t-5, 0] - self.inst["bid_prices"][abs_t-5, 0]) / (mid_prev + 1e-8)
            spread_change = spread - prev_spread
        else:
            momentum = 0.0
            spread_change = 0.0

        carry_frac = self.remaining_carry / (self.volume_to_fill + 1e-8)

        obs = np.array([
            remaining_vol_frac,
            remaining_time_frac,
            spread,
            mid_return,
            vol_imb_1,
            vol_imb_total,
            ask_depth,
            bid_depth,
            cum_is,
            momentum,
            spread_change,
            carry_frac,
        ], dtype=np.float32)

        obs = np.nan_to_num(obs, nan=0.0, posinf=10.0, neginf=-10.0)
        obs = np.clip(obs, -10.0, 10.0)
        return obs

    def step(self, action):
        position = self.action_multipliers[action] * self.twap_rate

        max_remaining = self.volume_to_fill - self.volume_allocated
        position = min(position, max(0.0, max_remaining))

        if self.t == self.K - 1:
            position = max(0.0, max_remaining)

        self.volume_allocated += position

        abs_t = self.start_second + self.t
        to_fill = position + self.remaining_carry
        self.remaining_carry = 0.0
        step_volume = 0.0
        step_cost = 0.0

        if to_fill > 1e-12:
            remaining = to_fill
            for level in range(5):
                price = self.inst["ask_prices"][abs_t, level]
                avail = self.inst["ask_vols"][abs_t, level]
                if math.isnan(price) or math.isnan(avail):
                    continue
                take = min(remaining, avail)
                step_cost += take * price
                step_volume += take
                remaining -= take
                if remaining <= 1e-12:
                    break
            self.remaining_carry = remaining

        self.volume_executed += step_volume
        self.value_executed += step_cost

        # === REWARD: Differential mark-to-market advantage over TWAP-K ===
        mid_t = (self.inst["ask_prices"][abs_t, 0] + self.inst["bid_prices"][abs_t, 0]) / 2
        if math.isnan(mid_t):
            mid_t = self.arrival_mid

        agent_remaining = self.volume_to_fill - self.volume_executed
        agent_total_cost = self.value_executed + agent_remaining * mid_t

        twap_remaining = self.volume_to_fill - self.twap_cum_vol[self.t]
        twap_total_cost = self.twap_cum_cost[self.t] + twap_remaining * mid_t

        normalizer = self.arrival_mid * self.volume_to_fill
        advantage = (twap_total_cost - agent_total_cost) / normalizer

        reward = (advantage - self.prev_advantage) * self.REWARD_SCALE
        self.prev_advantage = advantage

        reward = max(-100.0, min(100.0, reward))

        self.t += 1
        done = self.t >= self.K
        info = {}

        if done:
            fill_ratio = self.volume_executed / (self.volume_to_fill + 1e-8)
            if fill_ratio < 0.99:
                reward -= (1.0 - fill_ratio) * 100.0

            if self.volume_executed > 0:
                avg_price = self.value_executed / self.volume_executed
                close = self.inst["close"]
                bps = abs(avg_price - close) / close * 10000
                vol_penalty = min(100.0, self.volume_to_fill / self.volume_executed)
                info["bps"] = bps * vol_penalty
                info["fill_ratio"] = fill_ratio
            else:
                info["bps"] = 10000.0
                info["fill_ratio"] = 0.0

        obs = self._get_obs()
        return obs, reward, done, False, info


# === Sanity checks ===
test_env = TradeExecutionEnv(train_instruments, volume_to_fill, K_SECONDS)

obs, _ = test_env.reset()
twap_reward = 0
for _ in range(K_SECONDS):
    obs, reward, done, _, info = test_env.step(4)  # action 4 = 1.0x
    twap_reward += reward
    if done:
        break
print(f"TWAP policy (action=4, 1.0x): BPS={info.get('bps', 'N/A'):.4f}, "
      f"Fill={info.get('fill_ratio', 0):.2%}, Reward={twap_reward:.4f}")

obs, _ = test_env.reset()
skip_reward = 0
for _ in range(K_SECONDS):
    obs, reward, done, _, info = test_env.step(0)  # action 0 = 0.0x
    skip_reward += reward
    if done:
        break
print(f"Skip policy (action=0, 0.0x):  BPS={info.get('bps', 'N/A'):.4f}, "
      f"Fill={info.get('fill_ratio', 0):.2%}, Reward={skip_reward:.4f}")

obs, _ = test_env.reset()
fast_reward = 0
for _ in range(K_SECONDS):
    obs, reward, done, _, info = test_env.step(9)  # action 9 = 3.0x
    fast_reward += reward
    if done:
        break
print(f"Fast policy (action=9, 3.0x):  BPS={info.get('bps', 'N/A'):.4f}, "
      f"Fill={info.get('fill_ratio', 0):.2%}, Reward={fast_reward:.4f}")

obs, _ = test_env.reset()
rand_reward = 0
for _ in range(K_SECONDS):
    obs, reward, done, _, info = test_env.step(test_env.action_space.sample())
    rand_reward += reward
    if done:
        break
print(f"Random policy:                 BPS={info.get('bps', 'N/A'):.4f}, "
      f"Fill={info.get('fill_ratio', 0):.2%}, Reward={rand_reward:.4f}")

nan_count = 0
for _ in range(20):
    obs, _ = test_env.reset()
    for _ in range(K_SECONDS):
        if np.isnan(obs).any():
            nan_count += 1
        obs, _, done, _, _ = test_env.step(test_env.action_space.sample())
        if done:
            break
print(f"\nNaN check: {nan_count} NaN observations in 20 episodes (should be 0)")
print(f"Episode length: {K_SECONDS} steps | Actions: {len(test_env.action_multipliers)} "
      f"({', '.join(f'{m}x' for m in test_env.action_multipliers)})")

TWAP policy (action=4, 1.0x): BPS=2.2872, Fill=100.00%, Reward=0.0000
Skip policy (action=0, 0.0x):  BPS=10.4022, Fill=79.77%, Reward=-25.0489
Fast policy (action=9, 3.0x):  BPS=4.6839, Fill=100.00%, Reward=-0.0125
Random policy:                 BPS=0.9302, Fill=100.00%, Reward=1.7128

NaN check: 0 NaN observations in 20 episodes (should be 0)
Episode length: 20 steps | Actions: 10 (0.0x, 0.25x, 0.5x, 0.75x, 1.0x, 1.25x, 1.5x, 2.0x, 2.5x, 3.0x)


## Training

PPO (Proximal Policy Optimization) with a small MLP policy (2 hidden layers, 64 neurons each).
PPO uses trajectory rollouts rather than replay buffers, making it better suited for
environments with small per-step reward signals. RL-Exec (BTC-USD execution) used PPO
for this exact reason.

In [6]:
train_env = TradeExecutionEnv(train_instruments, volume_to_fill, K_SECONDS)

# Linear LR decay: 3e-4 → 0 over training
def linear_schedule(progress_remaining: float) -> float:
    return progress_remaining * 3e-4

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=linear_schedule,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs={"net_arch": [64, 64]},
    verbose=1,
    seed=SEED,
)

TOTAL_TIMESTEPS = 1_000_000
print(f"Training for {TOTAL_TIMESTEPS} timesteps (~{TOTAL_TIMESTEPS // K_SECONDS} episodes)...")
print(f"LR schedule: 3e-4 → 0 (linear decay)")
model.learn(total_timesteps=TOTAL_TIMESTEPS)
print("Training complete.")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Training for 1000000 timesteps (~50000 episodes)...
LR schedule: 3e-4 → 0 (linear decay)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 20       |
|    ep_rew_mean     | -0.0337  |
| time/              |          |
|    fps             | 500      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 20           |
|    ep_rew_mean          | -0.129       |
| time/                   |              |
|    fps                  | 433          |
|    iterations           | 2            |
|    time_elapsed         | 9            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0117239505 |
|    clip_fraction        | 0.0982       |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.29        |
|    explained_variance   | -0.0634      |
|    learning_r

## Evaluation

Run the trained agent on validation instruments and compare BPS against TWAP-K.  
Uses `simulate_walk_the_book()` for final BPS — same function as all other strategies.

In [7]:
def evaluate(model, instruments, volume_to_fill, K_seconds):
    """Evaluate RL agent vs multiple TWAP baselines on a set of instruments."""
    env = TradeExecutionEnv(instruments, volume_to_fill, K_seconds)
    rl_bps_list = []
    twap_k_bps_list = []
    twap_14_bps_list = []
    twap_60_bps_list = []
    rl_positions_all = []

    for i in range(len(instruments)):
        inst = instruments[i]
        ask_prices = inst["ask_prices"]
        ask_vols = inst["ask_vols"]
        bid_prices = inst["bid_prices"]
        bid_vols = inst["bid_vols"]
        close_price = inst["close"]

        # --- RL Agent ---
        obs, _ = env.reset(options={"instrument_idx": i})
        rl_positions = np.zeros(60)
        for step in range(K_seconds):
            action, _ = model.predict(obs, deterministic=True)

            position = env.action_multipliers[int(action)] * env.twap_rate
            max_remaining = volume_to_fill - env.volume_allocated
            position = min(position, max(0.0, max_remaining))
            if step == K_seconds - 1:
                position = max(0.0, max_remaining)

            abs_t = env.start_second + step
            rl_positions[abs_t] = position

            obs, _, done, _, _ = env.step(int(action))
            if done:
                break

        rl_positions_all.append(rl_positions)

        def compute_bps(positions):
            vol, avg = simulate_walk_the_book(
                positions, ask_prices, ask_vols, bid_prices, bid_vols
            )
            if vol > 0 and not np.isnan(avg):
                ie = abs(avg - close_price) / close_price * 10000
                vp = min(100.0, volume_to_fill / vol)
                return ie * vp
            return None

        # --- RL BPS ---
        bps = compute_bps(rl_positions)
        if bps is not None:
            rl_bps_list.append(bps)

        # --- TWAP-K ---
        twap_k_pos = np.zeros(60)
        twap_k_pos[-K_seconds:] = volume_to_fill / K_seconds
        bps = compute_bps(twap_k_pos)
        if bps is not None:
            twap_k_bps_list.append(bps)

        # --- TWAP-14 ---
        twap_14_pos = np.zeros(60)
        twap_14_pos[-14:] = volume_to_fill / 14
        bps = compute_bps(twap_14_pos)
        if bps is not None:
            twap_14_bps_list.append(bps)

        # --- TWAP-60 ---
        twap_60_pos = np.full(60, volume_to_fill / 60)
        bps = compute_bps(twap_60_pos)
        if bps is not None:
            twap_60_bps_list.append(bps)

    return (np.array(rl_bps_list), np.array(twap_k_bps_list),
            np.array(twap_14_bps_list), np.array(twap_60_bps_list),
            rl_positions_all)


rl_bps, twap_k_bps, twap_14_bps, twap_60_bps, rl_positions_all = evaluate(
    model, val_instruments, volume_to_fill, K_SECONDS
)

print(f"\n{'='*60}")
print(f"RESULTS: {data_asset} (volume={volume_to_fill})")
print(f"{'='*60}")
print(f"{'Strategy':<20} {'Mean BPS':>10} {'Median BPS':>12}")
print(f"{'-'*60}")
print(f"{'RL Agent':<20} {rl_bps.mean():>10.4f} {np.median(rl_bps):>12.4f}")
print(f"{'TWAP-' + str(K_SECONDS) + 's':<20} {twap_k_bps.mean():>10.4f} {np.median(twap_k_bps):>12.4f}")
print(f"{'TWAP-14s':<20} {twap_14_bps.mean():>10.4f} {np.median(twap_14_bps):>12.4f}")
print(f"{'TWAP-60s':<20} {twap_60_bps.mean():>10.4f} {np.median(twap_60_bps):>12.4f}")
print(f"{'-'*60}")

# Compare RL against each baseline
for name, baseline in [("TWAP-" + str(K_SECONDS), twap_k_bps),
                        ("TWAP-14", twap_14_bps),
                        ("TWAP-60", twap_60_bps)]:
    diff = baseline.mean() - rl_bps.mean()
    wins = (rl_bps < baseline).sum()
    pct = (rl_bps < baseline).mean() * 100
    label = "RL wins" if diff > 0 else "TWAP wins"
    print(f"RL vs {name:<8}: {diff:+.4f} bps ({label}), "
          f"RL wins {wins}/{len(rl_bps)} ({pct:.1f}%)")

print(f"{'='*60}")
print(f"Instruments evaluated: {len(rl_bps)}")
print(f"\nSubmission comparison (TWAP-14): 5.0781 BPS")


RESULTS: XRPUSDT (volume=13736.0)
Strategy               Mean BPS   Median BPS
------------------------------------------------------------
RL Agent                 3.1492       2.7821
TWAP-20s                 3.5627       3.3628
TWAP-14s                 3.5853       3.2105
TWAP-60s                 3.9919       3.4486
------------------------------------------------------------
RL vs TWAP-20 : +0.4135 bps (RL wins), RL wins 84/131 (64.1%)
RL vs TWAP-14 : +0.4361 bps (RL wins), RL wins 86/131 (65.6%)
RL vs TWAP-60 : +0.8427 bps (RL wins), RL wins 79/131 (60.3%)
Instruments evaluated: 131

Submission comparison (TWAP-14): 5.0781 BPS


## Policy Analysis

Examine what the agent learned: average position profile across instruments,
compared to TWAP allocation.

In [8]:
# Average position profile
positions_array = np.array(rl_positions_all)  # [num_instruments, 60]
avg_positions = positions_array.mean(axis=0)

# TWAP-K profile for comparison
twap_profile = np.zeros(60)
twap_profile[-K_SECONDS:] = volume_to_fill / K_SECONDS

print("RL Agent — Average position profile (last K seconds):")
print(f"  Volume in last {K_SECONDS}s:   {avg_positions[-K_SECONDS:].sum():.4f} ({avg_positions[-K_SECONDS:].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume in last 5s:    {avg_positions[-5:].sum():.4f} ({avg_positions[-5:].sum()/volume_to_fill*100:.1f}%)")
print(f"  Volume at last sec:   {avg_positions[59]:.4f} ({avg_positions[59]/volume_to_fill*100:.1f}%)")
print()
print(f"Per-second breakdown (last {K_SECONDS} seconds):")
start = 60 - K_SECONDS
for s in range(start, 60):
    rl_avg = avg_positions[s]
    twap_avg = twap_profile[s]
    bar = '#' * int(rl_avg / volume_to_fill * 100)
    print(f"  t={s:2d}: RL={rl_avg:7.2f} TWAP={twap_avg:7.2f}  {bar}")
print()
print(f"Top 5 seconds by average position:")
top5 = np.argsort(avg_positions)[-5:][::-1]
for s in top5:
    print(f"  Second {s}: {avg_positions[s]:.4f} ({avg_positions[s]/volume_to_fill*100:.1f}%)")

RL Agent — Average position profile (last K seconds):
  Volume in last 20s:   13736.0000 (100.0%)
  Volume in last 5s:    2941.1817 (21.4%)
  Volume at last sec:   655.3435 (4.8%)

Per-second breakdown (last 20 seconds):
  t=40: RL= 791.65 TWAP= 686.80  #####
  t=41: RL= 740.54 TWAP= 686.80  #####
  t=42: RL= 688.11 TWAP= 686.80  #####
  t=43: RL= 720.88 TWAP= 686.80  #####
  t=44: RL= 757.58 TWAP= 686.80  #####
  t=45: RL= 705.15 TWAP= 686.80  #####
  t=46: RL= 715.64 TWAP= 686.80  #####
  t=47: RL= 671.07 TWAP= 686.80  ####
  t=48: RL= 768.06 TWAP= 686.80  #####
  t=49: RL= 665.83 TWAP= 686.80  ####
  t=50: RL= 715.64 TWAP= 686.80  #####
  t=51: RL= 705.15 TWAP= 686.80  #####
  t=52: RL= 737.92 TWAP= 686.80  #####
  t=53: RL= 761.51 TWAP= 686.80  #####
  t=54: RL= 650.10 TWAP= 686.80  ####
  t=55: RL= 678.94 TWAP= 686.80  ####
  t=56: RL= 642.24 TWAP= 686.80  ####
  t=57: RL= 500.68 TWAP= 686.80  ###
  t=58: RL= 463.98 TWAP= 686.80  ###
  t=59: RL= 655.34 TWAP= 686.80  ####

Top 5 se

## Generate Test Positions

Run the agent on test data and save positions in the submission format.

In [9]:
# Generate positions for validation instruments (demo — replace with test data when available)
print("Generating positions for validation instruments...")

times = Y_raw["time_in_hour"].sort_values().unique()

test_positions_df = pd.DataFrame()
env = TradeExecutionEnv(val_instruments, volume_to_fill, K_SECONDS)

for i, inst in enumerate(val_instruments):
    obs, _ = env.reset(options={"instrument_idx": i})
    positions = np.zeros(60)
    for step in range(K_SECONDS):
        action, _ = model.predict(obs, deterministic=True)
        pos = env.action_multipliers[int(action)] * env.twap_rate
        max_rem = volume_to_fill - env.volume_allocated
        pos = min(pos, max(0.0, max_rem))
        if step == K_SECONDS - 1:
            pos = max(0.0, max_rem)

        abs_t = env.start_second + step
        positions[abs_t] = pos
        obs, _, done, _, _ = env.step(int(action))
        if done:
            break

    df = pd.DataFrame({
        'anonymized_id': inst['id'],
        'time_in_hour': times,
        'position': positions,
    })
    test_positions_df = pd.concat([test_positions_df, df], ignore_index=True)

base_path = Path(f"positions/{data_asset}")
base_path.mkdir(parents=True, exist_ok=True)
target = base_path / "rl_agent.parquet"
test_positions_df.to_parquet(target, index=False, engine='pyarrow')
print(f"Saved {len(test_positions_df)} rows to {target}")

Generating positions for validation instruments...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Saved 7860 rows to positions/XRPUSDT/rl_agent.parquet
